# DocTamper 蒸馏训练 (Colab)

在 Colab 上运行轻量模型蒸馏训练，数据与依赖加载方式与根目录 `DTD Reproduction on DocTamper.ipynb` 一致。

## 1. 挂载 Drive 并克隆仓库

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive/')

!git clone -b feat-distillation https://github.com/LeSiIence/DocTamper.git
%cd DocTamper

## 2. 安装依赖

在项目根目录安装，便于 `train_distill.py` 和 `dataloader` 导入。Colab 已带 PyTorch，仅安装缺失包与 jpegio。

In [ ]:
!pip install -q lmdb albumentations segmentation_models_pytorch timm efficientnet_pytorch tqdm
!pip install -q opencv-python-headless Pillow

# jpegio 从源码安装（与根目录 ipynb 一致）
!git clone https://github.com/dwgoon/jpegio.git
%cd jpegio
!python setup.py install -q
%cd ..

## 3. 从 Drive 拷贝数据与权重

需在 Drive 中准备：
- `DocTamperV1-FCD`：LMDB 数据集目录
- `checkpoints`：教师权重等（内含 `dtd_doctamper.pth`）

与 DTD 笔记本一致：`qt_table.pk`、`pks`（压缩记录目录，含 `DocTamperV1-FCD_75.pk` 等）应在 clone 下来的代码目录中，无需从 Drive 拷贝。

In [ ]:
import os

# 设定 TargetFolder 为 MyDrive 下的根数据目录
TargetFolder = '/content/drive/MyDrive/TargetFolder'

# 修复：去掉了前面的 f，并在外层使用引号（防止路径中存在空格导致命令解析错误）
!cp -r "{TargetFolder}/DocTamperV1-FCD" './'
assert os.path.exists('DocTamperV1-FCD'), 'DocTamperV1-FCD not found in TargetFolder'

# qt_table.pk 在 clone 下来的代码目录中（与 DTD 笔记本一致，见 DocTamper 根目录）
assert os.path.exists('qt_table.pk'), 'qt_table.pk not found in clone directory (expected under DocTamper/qt_table.pk)'

# pks 在 clone 下来的代码目录中，无需从 Drive 拷贝
assert os.path.exists('pks'), 'pks not found in clone directory (expected under DocTamper/pks)'

os.makedirs('pths', exist_ok=True)
!cp -r "{TargetFolder}/checkpoints" './pths'
assert os.path.exists('pths/checkpoints'), 'pths/checkpoints not found in TargetFolder' # 这里的断言路径也修正了一下，匹配上文的复制逻辑

# 教师权重放到 pths 根下
!cp pths/checkpoints/dtd_doctamper.pth pths/ 2>/dev/null || true
print('Data and checkpoints ready.')

## 4. 启动蒸馏训练

默认使用 `DocTamperV1-FCD`、教师权重 `pths/dtd_doctamper.pth`，可按需改参数。

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python train_distill.py \
  --data_root . \
  --lmdb_name DocTamperV1-FCD \
  --teacher_pth pths/dtd_doctamper.pth \
  --minq 75 \
  --epochs 50 \
  --batch_size 8 \
  --lr 1e-4 \
  --save_dir pths \
  --save_interval 5

## 5. （可选）将 checkpoint 拷回 Drive

In [ ]:
# 训练结束后，把 pths 下的 light_dtd_distill_*.pth 拷回 Drive 保存
!cp pths/light_dtd_distill_*.pth '/content/drive/MyDrive/' 2>/dev/null || echo 'No light_dtd_distill checkpoints found.'